# Suricata EVE JSON Sanitization

This notebook sanitizes Suricata `eve.json` logs using the same policy as the other cleaners:
- **Usernames:** pseudonymize **only** `newt` (case-insensitive). All other names are left unchanged.
- **FQDNs:** pseudonymize **only** Azure/Azure-hosted domains (suffix allowlist). Public non-Azure domains are preserved.
- **IPs/ports/protocols/timestamps:** preserved to keep security semantics.

The sanitizer operates line-by-line (JSONL): each line is parsed as JSON, sanitized, and written back as JSON.

In [2]:
import json
import hashlib
import ipaddress
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple


In [3]:
# ------------------------
# Configuration
# ------------------------
BASE_DIR = Path.cwd().parent.resolve().joinpath("data")

# Input file(s) to find per scenario (recursive)
FILES = {
    "suricata": "eve.json",
}

def extract_scenario(path: Path) -> str:
    """Extract scenario id like 'sc1' from any part of the path."""
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """Return inputs[scenario][logtype] -> list[Path]."""
    inputs = defaultdict(lambda: defaultdict(list))
    for logtype, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][logtype].append(p)
    return inputs

INPUTS = collect_inputs(BASE_DIR, FILES)

# Quick sanity print: how many matches per scenario/logtype
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")

OUT_DIR = BASE_DIR.parent.resolve().joinpath("sanidata")
MAP_DIR = BASE_DIR.parent.resolve().joinpath("mappings")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

SALT_PATH = MAP_DIR / "salt.txt"  # stored locally; do NOT publish

# newt-only user allowlist (kept for policy parity; Suricata typically has no users)
ONLY_PSEUDONYMIZE_USERS = {"newt"}  # lowercase allowlist

# Azure/Azure-hosted FQDN suffix allowlist (suffix match == wildcard *.suffix)
AZURE_FQDN_SUFFIXES = (
    "trafficmanager.net",
    "azure.com",
    "azure.net",
    "azureedge.net",
    "cloudapp.azure.com",
    "azurewebsites.net",
    "windows.net",
    "database.windows.net",
    "redis.cache.windows.net",
    "servicebus.windows.net",
    "vault.azure.net",
    "azure-api.net",
    "core.windows.net",  # optional; covered by windows.net but kept for clarity
)

def _is_empty(x) -> bool:
    return x is None or (isinstance(x, str) and x.strip() == "")


sc3: {'suricata': 1}
sc4: {'suricata': 1}
sc7: {'suricata': 1}


In [4]:
# ------------------------
# Stable mapping utilities (salt + persisted JSON per category)
# ------------------------
def load_or_create_salt(path: Path) -> bytes:
    """Load a stable secret salt, creating it if missing."""
    if path.exists():
        return path.read_bytes().strip()
    # Create a new random salt (32 bytes) and store it locally.
    import os
    s = os.urandom(32)
    path.write_bytes(s.hex().encode("utf-8"))
    return s.hex().encode("utf-8")

SALT = load_or_create_salt(SALT_PATH)

def _map_path(map_dir: Path, name: str) -> Path:
    return map_dir / f"{name}.json"

def load_map(map_dir: Path, name: str) -> Dict[str, str]:
    p = _map_path(map_dir, name)
    if not p.exists():
        return {}
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        # If the mapping is corrupted, fail loudly rather than silently.
        raise

def save_map(map_dir: Path, name: str, d: Dict[str, str]) -> None:
    p = _map_path(map_dir, name)
    p.write_text(json.dumps(d, indent=2, sort_keys=True), encoding="utf-8")

def map_token(value: str, prefix: str, map_name: str, digits: int = 4) -> str:
    """Deterministically map a value to a compact token, persisted in MAP_DIR.

    - Uses SHA-256 over SALT||value to derive a candidate numeric id.
    - Resolves rare collisions deterministically.
    - Stores mapping in mappings/<map_name>.json.
    """
    if _is_empty(value):
        return value

    v = str(value)
    m = load_map(MAP_DIR, map_name)

    # Use normalized key for stable mapping (case-insensitive where appropriate).
    key = v

    if key in m:
        return m[key]

    used = set(m.values())
    base = 10 ** digits

    # Collision-safe token generation: try different digest slices.
    digest = hashlib.sha256(SALT + b"||" + key.encode("utf-8", errors="ignore")).hexdigest()

    for offset in range(0, len(digest) - 8, 8):
        chunk = digest[offset:offset + 8]
        num = int(chunk, 16) % base
        tok = f"{prefix}_{num:0{digits}d}"
        if tok not in used:
            m[key] = tok
            save_map(MAP_DIR, map_name, m)
            return tok

    # Extremely unlikely fallback: append a counter.
    for i in range(1, 10000):
        digest2 = hashlib.sha256(SALT + b"||" + key.encode("utf-8", errors="ignore") + f"||{i}".encode()).hexdigest()
        chunk = digest2[:8]
        num = int(chunk, 16) % base
        tok = f"{prefix}_{num:0{digits}d}"
        if tok not in used:
            m[key] = tok
            save_map(MAP_DIR, map_name, m)
            return tok

    raise RuntimeError("Unable to allocate a unique token (unexpected).")


In [5]:
# ------------------------
# Azure-only FQDN sanitization helpers
# ------------------------
def is_ip_literal(s: str) -> bool:
    """True if s is an IPv4/IPv6 literal."""
    try:
        ipaddress.ip_address(s)
        return True
    except Exception:
        return False

def is_azure_fqdn(fqdn: str) -> bool:
    """Suffix match (equivalent to wildcard *.suffix)."""
    if _is_empty(fqdn):
        return False
    f = str(fqdn).strip().lower().rstrip(".")
    return any(f == sfx or f.endswith("." + sfx) for sfx in AZURE_FQDN_SUFFIXES)

def sanitize_fqdn_value(v: str) -> str:
    """Pseudonymize only Azure/Azure-hosted FQDNs; keep everything else intact."""
    if _is_empty(v):
        return v
    s = str(v).strip()
    if is_ip_literal(s):
        return v

    # Preserve trailing dot (rooted DNS names)
    trailing_dot = s.endswith(".")
    core = s.rstrip(".")

    if is_azure_fqdn(core):
        tok = map_token(core.lower(), "FQDN", "fqdn_map", digits=4)
        out = f"{tok}.example"
        return out + ("." if trailing_dot else "")
    return v


In [6]:
# ------------------------
# Suricata EVE sanitizer (JSONL)
# ------------------------
FQDN_KEYS = {
    # Common fields in Suricata EVE:
    "hostname",  # http.hostname
    "rrname",    # dns.rrname
    "rdata",     # dns.answers[].rdata (may be IP or FQDN)
    "sni",       # tls.sni
    "mname",     # dns.authorities[].soa.mname
    "rname",     # dns.authorities[].soa.rname
}

def sanitize_obj(obj: Any) -> Any:
    """Recursively sanitize nested dict/list structures."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(v, str) and k in FQDN_KEYS:
                out[k] = sanitize_fqdn_value(v)
            elif k == "grouped" and isinstance(v, dict):
                # grouped can contain lists of FQDNs/IPs under keys like CNAME/A/AAAA
                out[k] = {gk: [sanitize_fqdn_value(x) if isinstance(x, str) else x for x in gv]
                          if isinstance(gv, list) else gv
                          for gk, gv in v.items()}
            else:
                out[k] = sanitize_obj(v)
        return out

    if isinstance(obj, list):
        return [sanitize_obj(x) for x in obj]

    # Primitive (int/float/bool/None/other) or strings on non-FQDN keys
    return obj

def sanitize_eve_line(line: str) -> str:
    """Sanitize a single EVE JSON line. If parsing fails, return the original line."""
    if _is_empty(line):
        return line
    try:
        evt = json.loads(line)
    except Exception:
        return line  # keep raw if not valid JSON

    evt2 = sanitize_obj(evt)
    return json.dumps(evt2, ensure_ascii=False, separators=(",", ":"))


In [7]:
# ------------------------
# Processing: sanitize all matched eve.json per scenario
# ------------------------
def output_path_for(input_path: Path, scenario: str, out_root: Path) -> Path:
    """Write outputs under: out_root/<scenario>/<filename>."""
    sc_dir = out_root / scenario
    sc_dir.mkdir(parents=True, exist_ok=True)
    return sc_dir / input_path.name

def sanitize_jsonl_file(in_path: Path, out_path: Path) -> Tuple[int, int]:
    """Sanitize a JSONL file. Returns (lines_processed, json_parse_failures)."""
    n = 0
    bad = 0
    with in_path.open("r", encoding="utf-8", errors="replace") as fin,          out_path.open("w", encoding="utf-8") as fout:
        for raw in fin:
            line = raw.rstrip("\n")
            sanitized = sanitize_eve_line(line)
            if sanitized == line:
                # heuristic: if it didn't change, it might still be valid; we only count failures
                # by re-parsing quickly (cheap enough, Suricata lines are per-event).
                try:
                    json.loads(line)
                except Exception:
                    bad += 1
            fout.write(sanitized + "\n")
            n += 1
    return n, bad

total_lines = 0
total_bad = 0

for sc in sorted(INPUTS.keys()):
    for in_eve in INPUTS[sc].get("suricata", []):
        out_eve = output_path_for(in_eve, sc, OUT_DIR)
        n, bad = sanitize_jsonl_file(in_eve, out_eve)
        total_lines += n
        total_bad += bad
        print(f"[{sc}] suricata: {in_eve} -> {out_eve} ({n} lines, {bad} parse failures)")

print(f"Done. Total lines: {total_lines}, total parse failures: {total_bad}")
print(f"Outputs root: {OUT_DIR.resolve()}")
print(f"Mappings stored in: {MAP_DIR.resolve()}")


[sc3] suricata: /Users/zhuoran/Projects/SSCMDataset/data/sc3/eve.json -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc3/eve.json (65385 lines, 0 parse failures)
[sc4] suricata: /Users/zhuoran/Projects/SSCMDataset/data/sc4/eve.json -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc4/eve.json (116215 lines, 0 parse failures)
[sc7] suricata: /Users/zhuoran/Projects/SSCMDataset/data/sc7/eve.json -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/eve.json (58880 lines, 0 parse failures)
Done. Total lines: 240480, total parse failures: 0
Outputs root: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mappings stored in: /Users/zhuoran/Projects/SSCMDataset/mappings


In [8]:
# ------------------------
# Preview: show the first N lines of each sanitized file
# ------------------------
def head(path: Path, n: int = 5):
    """Print the first n lines of a text file."""
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip("\n"))

for sc in sorted(INPUTS.keys()):
    for in_eve in INPUTS[sc].get("suricata", []):
        out_eve = output_path_for(in_eve, sc, OUT_DIR)
        print(f"\n=== Scenario {sc} | {out_eve} ===")
        if out_eve.exists():
            head(out_eve, 5)
        else:
            print("(missing)")



=== Scenario sc3 | /Users/zhuoran/Projects/SSCMDataset/sanidata/sc3/eve.json ===
{"timestamp":"2025-12-14T06:16:15.290604-0500","flow_id":2071267333596670,"in_iface":"eth0","event_type":"http","src_ip":"10.0.0.4","src_port":41398,"dest_ip":"168.63.129.16","dest_port":80,"proto":"TCP","pkt_src":"wire/pcap","tx_id":0,"http":{"hostname":"168.63.129.16","url":"/machine/?comp=goalstate","http_user_agent":"WALinuxAgent/2.15.0.1","http_content_type":"text/xml","http_method":"GET","protocol":"HTTP/1.1","status":200,"length":2102}}
{"timestamp":"2025-12-14T06:16:15.290786-0500","flow_id":2071267333596670,"in_iface":"eth0","event_type":"fileinfo","src_ip":"168.63.129.16","src_port":80,"dest_ip":"10.0.0.4","dest_port":41398,"proto":"TCP","pkt_src":"wire/pcap","http":{"hostname":"168.63.129.16","url":"/machine/?comp=goalstate","http_user_agent":"WALinuxAgent/2.15.0.1","http_content_type":"text/xml","http_method":"GET","protocol":"HTTP/1.1","status":200,"length":2102},"app_proto":"http","fileinfo"